# Module 1 · Lab 1.1 — Multi-Provider LLM Setup & Structured Outputs
## Engineering reliable LLM calls across OpenAI · Anthropic · Google · Groq

---

### Lab Overview

In a chat window you *judge* models by reading their answers. Now you **engineer** against them. Real applications don't paste replies into a chat window — they call models **programmatically**, need **structured data** they can route and store, and must stay up even when a provider has a bad day.

This lab works across **four providers** — OpenAI, Anthropic, Google and **Groq** (serving open-weight models such as **Qwen** on custom inference hardware) — and turns ad-hoc calls into production patterns:

1. The **native shape** of all four SDKs (Responses, Messages, GenAI, OpenAI-compatible)
2. **Unified wrapper functions** so the rest of your code is provider-agnostic
3. **Temperature vs. reasoning effort** — the two knobs people confuse most
4. **Structured outputs with Pydantic** — get a *validated object*, not free text *(the headline skill)*
5. **Cost & latency** benchmarking — the data behind a vendor decision
6. A **multi-provider fallback** for production reliability

> **Judging vs. building.** Reading answers = explore & judge models *qualitatively* (free text, subjective). **This lab = build against them**: machine-readable outputs, measured latency/cost, and reliability patterns. The metric here is **schema-conformance, tokens, cost, and uptime** — not "which answer reads nicer."

### What you'll learn
- Call 4 providers natively, then behind one interface
- Control randomness (temperature) vs. depth (reasoning effort)
- Force LLM output into a **Pydantic schema** and validate it
- Estimate cost at scale and add a **fallback chain**

### Prerequisites
- API keys for **OpenAI, Anthropic, Google (Gemini),** and **Groq** (Groq has a free tier)
- Colab (recommended) or local `.env`

## 1. Install the SDKs

In [ ]:
# 📦 Packages this notebook needs. On the training VM they are already installed,
# so the install line below stays commented out.
# Running somewhere else (Colab, your own laptop)? Uncomment it and run this cell once.
# !pip install -qU anthropic google-genai groq openai pandas pydantic python-dotenv

## 2. Load API keys — securely

The standard Colab-Secrets-first / `.env`-fallback loader, here with **four** keys. On Colab add them in the 🔑 **Secrets** panel; locally put them in `.env`.

In [1]:
import os

def load_keys(required):
    """Load API keys from Colab Secrets (on Colab) or a local .env file (elsewhere)."""
    try:
        from google.colab import userdata          # running on Google Colab
        for k in required:
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass                                 # not set in Colab Secrets
        source = "Colab Secrets"
    except ImportError:                              # running locally / non-Colab
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv(usecwd=True))     # searches cwd upward for a .env
        except ImportError:
            pass
        source = ".env / environment"
    print(f"\U0001F511 Key source: {source}")
    for k in required:
        print(f"   {k:<22} {'\u2705' if os.environ.get(k) else '\u274C MISSING'}")

load_keys(["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"])

🔑 Key source: .env / environment
   OPENAI_API_KEY         ✅
   ANTHROPIC_API_KEY      ✅
   GEMINI_API_KEY         ✅
   GROQ_API_KEY           ✅


## 3. Initialize all four clients

Each provider exposes a different API surface. Learn the **native** shape of each before we hide it behind a wrapper.

| Provider | Client | Primary call | Notable trait |
|---|---|---|---|
| OpenAI | `OpenAI()` | `responses.create()` / `chat.completions.create()` | Reasoning models + native structured output |
| Anthropic | `Anthropic()` | `messages.create()` | `system` is a top-level arg, not a message |
| Google | `genai.Client()` | `models.generate_content()` | Config goes in a `GenerateContentConfig` object |
| Groq | `Groq()` | `chat.completions.create()` | **OpenAI-compatible** API; very low latency |

In [2]:
import time
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types
from groq import Groq

openai_client    = OpenAI()
anthropic_client = Anthropic()
gemini_client    = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
groq_client      = Groq()

# Pinned model ids — verify against current provider docs before each run.
M = {
    "openai_reason": "gpt-5-mini",            # reasoning model (Responses API)
    "openai_chat":   "gpt-4.1-mini",          # fast non-reasoning chat model
    "anthropic":     "claude-haiku-4-5",      # fast Claude (Sonnet 4.6 / Opus 4.8 for harder work)
    "gemini":        "gemini-2.5-flash",      # Google cost leader
    "groq":          "qwen/qwen3.8-27b",         # open-weight Qwen on Groq
}
print("All 4 clients initialized:", list(M.values()))

All 4 clients initialized: ['gpt-5-mini', 'gpt-4.1-mini', 'claude-haiku-4-5', 'gemini-2.5-flash', 'qwen/qwen3.8-27b']


## 4. "Hello" from each provider — one business task each

Four providers, four industries, four native SDK shapes. Read how each call is structured.

### 4a · OpenAI — Insurance claim intake 🏥 *(Responses API)*

The **Responses API** is OpenAI's newer interface and the home of reasoning models. `instructions` is the system role; `reasoning={"effort": ...}` controls how hard the model thinks.

In [3]:
resp = openai_client.responses.create(
    model=M["openai_reason"],
    instructions="You are a senior claims assistant at a health-insurance company. Be precise and empathetic.",
    input="A customer writes: 'I was hospitalized for 2 days with dengue. How do I file a cashless claim?' "
          "Give a short, numbered step-by-step.",
    reasoning={"effort": "low"},
)
print(resp.output_text)

Sorry to hear you were hospitalized — I hope you’re recovering. To file a cashless claim, follow these steps:

1. Notify the insurer immediately (or within 24 hours of admission) using the 24x7 claims/helpline, mobile app, or website portal. If you’re at a network hospital, the hospital’s TPAs/insurance desk can notify for you.  
2. Confirm the hospital is part of your insurer’s cashless network. If not, ask the insurer about options (pre-approval, reimbursement, or transfer).  
3. Provide policy details: policy number, insured name, contact number and member ID/card. Have a photo or copy of the policy handy.  
4. Submit preliminary documents the insurer asks for (you or the hospital can): ID proof, policy card, filled pre-authorization form, admission note, treating doctor’s recommendation, and any immediate test reports.  
5. Hospital forwards the pre-authorization request and clinical notes to the insurer/TPA. The insurer reviews and issues an authorization number if approved (usual

In [ ]:
print(resp)  # entire repsonse object, including reasoning trace and token usage

Response(id='resp_0e447ad07a761fa6006acc915f1ae087d1980ad2d785124417', created_at=1791791455.0, error=None, incomplete_details=None, instructions='You are a senior claims assistant at a health-insurance company. Be precise and empathetic.', metadata={}, model='gpt-5-mini-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0e447ad07a761fa6006acc91602b2c87d19a4ce1cd457f9d14', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqzJFjwJpiCxaWDrkOJaiWnbVi4z3zDMkyvLnKqOTnji1gSVDXnwY9gIL2PZxATojeXNAT9GpMUwvXN9uv4Znpp1rW3vgFoxgU1NPuflBdBuBSACSM7PG-oy1Xhp5CBFc_8hpZWvdYu68eF4n-HeXSKGdtQ1SFMNfV31oP_scM2KX-U-8lOHfoVQwWkRFf2xXIGOIwGozhGjuUgVWlBqYK0a1Zdvjf6yn8G9PQZ1b11nQWMUQVaKNK-xkuEGw6aRgpvne9K-4C-uxpYMu0KtS1nw14lOh7YF_QJO_Tm7xjHoZA3CkVWHl__U1W7E0mg3uZpzU2SV5UGIYl7xZ5Z2MLLyITwQAAMgBDbTx033xUIHpCmETvdXyDa51LaRMDkKNJDAYBmh_YB5dZvAmUxQSKDN1j5pgePO6K_fU23pzxgWNB0XLk9z0HkOH4Ul_g03W_HDklLDHi2AtmLqueQsvsRudREaokkZCB0gixol4pSRdG5Q-OkCDc7eK7Pa-71vzXyd1jsxmj7z4wpkXT_yeCATMci

### 4b · Anthropic — Patient-facing health FAQ 🩺 *(Messages API)*

Claude's `messages.create()` puts the **system prompt as a top-level argument** (not inside `messages`) and **requires `max_tokens`**.

In [5]:
msg = anthropic_client.messages.create(
    model=M["anthropic"],
    max_tokens=400,
    system="You are a patient-friendly hospital assistant. Use simple, reassuring language. You never give a diagnosis.",
    messages=[{"role": "user", "content": "I have a fasting blood-sugar test at 8 AM tomorrow. Can I drink water before it?"}],
)
print(msg.content[0].text)

Yes, you can drink water! Water is perfectly fine to have before a fasting blood sugar test.

When your doctor says to "fast," it typically means:
- **No food** for 8-12 hours before the test
- **No sugary drinks** (juice, soda, etc.)
- **No coffee or tea** with milk or sugar
- **Water is okay** - drink as much as you need

In fact, staying hydrated with water is actually helpful for the blood draw itself, as it makes it easier for the nurse to find your vein.

If you have any specific questions about your test preparation, it's always a good idea to check with your doctor's office to confirm their exact instructions, just to be sure.


### 4c · Google Gemini — E-commerce product copy 🛍️ *(GenAI SDK)*

Gemini takes the prompt as `contents` and **all settings inside a `GenerateContentConfig`** object.

In [6]:
r = gemini_client.models.generate_content(
    model=M["gemini"],
    contents="Write a 2-sentence product description for a 1L stainless-steel insulated water bottle that keeps "
             "drinks cold for 24 hours. Upbeat and benefits-first.",
    config=types.GenerateContentConfig(system_instruction="You are a creative e-commerce copywriter."),
)
print(r.text)

Quench your thirst with ultimate refreshment that lasts! Our 1L stainless steel insulated water bottle keeps your drinks perfectly ice-cold for an incredible 24 hours.


### 4d · Groq — Logistics status summary 🚚 *(open-weight Qwen, OpenAI-compatible)*

Groq runs open-weight models (such as Qwen) on custom hardware and exposes an **OpenAI-compatible** API — if you know `chat.completions.create()`, you already know Groq.

In [7]:
r = groq_client.chat.completions.create(
    model=M["groq"],
    messages=[
        {"role": "system", "content": "You are a logistics assistant. Summarize shipment events into ONE clear customer-facing line."},
        {"role": "user",   "content": "Events: [Picked up Mumbai 10:02] [In transit Pune 14:20] "
                                       "[Out for delivery Bangalore 08:15] [Delivery attempted — customer unavailable 11:40]. "
                                       "Summarize the current status."},
    ],
)
print(r.choices[0].message.content)

Your package in Bangalore had a delivery attempt later this morning, but the driver could not reach a recipient.


## 5. Unified wrapper functions — one interface, four providers

The native shapes differ, but our app shouldn't care. Each wrapper takes `(system, user)` and returns the **same dict**: `{provider, model, text, latency, tokens}`. This provider-agnostic pattern is what makes the comparison, cost analysis, and fallback below trivial.

In [10]:
def call_openai(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["openai_chat"]
    t0 = time.time()
    r = openai_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "OpenAI", "model": model, "text": r.choices[0].message.content,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.total_tokens}

def call_anthropic(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["anthropic"]
    t0 = time.time()
    r = anthropic_client.messages.create(
        model=model, max_tokens=max_tokens, temperature=temperature,
        system=system, messages=[{"role": "user", "content": user}])
    return {"provider": "Anthropic", "model": model, "text": r.content[0].text,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.input_tokens + r.usage.output_tokens}

def call_gemini(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["gemini"]
    t0 = time.time()
    r = gemini_client.models.generate_content(
        model=model, contents=user,
        config=types.GenerateContentConfig(system_instruction=system, temperature=temperature, max_output_tokens=max_tokens))
    return {"provider": "Google", "model": model, "text": r.text,
            "latency": round(time.time() - t0, 2), "tokens": r.usage_metadata.total_token_count}

def call_groq(system, user, model=None, temperature=0.7, max_tokens=500):
    model = model or M["groq"]
    t0 = time.time()
    r = groq_client.chat.completions.create(
        model=model, temperature=temperature, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return {"provider": "Groq", "model": model, "text": r.choices[0].message.content,
            "latency": round(time.time() - t0, 2), "tokens": r.usage.total_tokens}

# PROVIDERS = {"OpenAI": call_openai, "Anthropic": call_anthropic, "Google": call_gemini, "Groq": call_groq}
PROVIDERS = {"OpenAI": call_openai, "Groq": call_groq}
print("Unified wrappers ready:", list(PROVIDERS))

Unified wrappers ready: ['OpenAI', 'Groq']


## 6. Side-by-side comparison — same prompt, all four 🏦 *(Banking)*

**Business scenario.** A bank's app must explain a *declined transaction* to a worried customer — calmly and clearly. We send the **same** system+user prompt to all four providers and capture latency and token usage. This is the quantitative comparison that reading answers alone leaves out.

In [11]:
SYSTEM = "You are a bank's customer-support assistant. Explain clearly and calmly, in under 80 words."
USER   = ("My debit card was just declined at a petrol pump even though I have ₹40,000 in my account. "
          "Why would this happen and what should I do right now?")

results = {}
for name, fn in PROVIDERS.items():
    try:
        results[name] = fn(SYSTEM, USER)
    except Exception as e:
        results[name] = {"provider": name, "model": "-", "text": f"[ERROR] {e}", "latency": None, "tokens": None}
    print(f"\n=== {name} ({results[name]['model']}) ===")
    print(results[name]["text"])


=== OpenAI (gpt-4.1-mini) ===
Your debit card might be declined due to reasons like daily withdrawal limits, a temporary hold, or network issues. Sometimes, petrol pumps require pre-authorization holds that exceed your limit. Please check your account for alerts, ensure your card is active, and try another payment method. To resolve this quickly, contact your bank’s customer support to verify your card status and lift any holds.

=== Groq (qwen/qwen3.8-27b) ===
Common causes include a temporary system glitch, insufficient available balance due to holds, or a security flag.

To resolve this:
1. Check the banking app for a "Card Declined" notification or specific error code.
2. Verify your available balance isn't locked by recent transactions.
3. Ensure your card is active and not expired.
4. If the issue persists, contact our helpline immediately for a security review or card reissue. We apologize for the inconvenience.


In [12]:
import pandas as pd

df = pd.DataFrame([{"Provider": r["provider"], "Model": r["model"],
                    "Latency (s)": r["latency"], "Tokens": r["tokens"]} for r in results.values()])
print(df.to_string(index=False))
print("\nGroq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.")

Provider            Model  Latency (s)  Tokens
  OpenAI     gpt-4.1-mini         1.56     141
    Groq qwen/qwen3.8-27b         0.43     175

Groq usually wins latency (custom inference HW). Token counts differ — each provider tokenizes differently.


## 7. Temperature vs. Reasoning Effort — the two knobs people confuse

| Knob | Applies to | Controls | Range |
|---|---|---|---|
| **Temperature** | non-reasoning models (`gpt-4.1-mini`, `claude-haiku-4-5`, `qwen3.8-27b`) | randomness / creativity | `0.0` (deterministic) → `2.0` (wild) |
| **Reasoning effort** | reasoning models (`gpt-5-mini`) | depth of internal "thinking" | `low` / `medium` / `high` |

**Key rule:** reasoning models typically **don't take a temperature** — you steer them with *effort*, not randomness.

### 7a · Temperature changes *randomness* (non-reasoning model)

In [13]:
prompt = "Write a 1-sentence tagline for a budgeting app aimed at college students."
for t in (0.1, 0.7, 1.2):
    out = call_openai("You are a witty copywriter.", prompt, temperature=t)
    print(f"temp={t}: {out['text']}")

temp=0.1: Budget smarter, party harder — your wallet’s new BFF for college cash control.
temp=0.7: Budget smarter, party harder – your wallet’s new BFF for college life!
temp=1.2: Budget better, party smarter—because your ramen deserves an upgrade.


### 7b · Same prompt, 3 runs at high temperature → different every time

In [16]:
for i in range(3):
    out = call_openai("You are a witty copywriter.", prompt, temperature=0.1)
    print(f"run {i+1}: {out['text']}")

run 1: Budget smarter, party harder — your college cash coach in your pocket.
run 2: Budget smarter, party harder—your wallet’s new BFF for college cash control.
run 3: Budget smart, party harder—your cash’s new BFF for college life.


In [17]:
for i in range(3):
    out = call_openai("You are a witty copywriter.", prompt, temperature=1.2)
    print(f"run {i+1}: {out['text']}")

run 1: Save bucks, smash goals—budgeting made simple for college hustlers.
run 2: Budget smarter, party harder—your college cash companion.
run 3: Spend smart, stress less—budget like a boss, even between classes.


### 7c · Reasoning effort changes *depth* (reasoning model, deterministic)

In [19]:
def openai_reasoning(prompt, effort="medium"):
    t0 = time.time()
    r = openai_client.responses.create(model=M["openai_reason"], input=prompt, reasoning={"effort": effort})
    return r.output_text, round(time.time() - t0, 2)

q = ("A retail customer asks why diversification reduces risk in a mutual-fund portfolio. "
     "Explain clearly, including the basic math intuition.")
for effort in ("low", "medium", "high"):
    text, lat = openai_reasoning(q, effort)
    print(f"\n--- effort={effort}  ({lat}s) ---\n{text}...")


--- effort=low  (8.2s) ---
Short answer
Diversification reduces a portfolio’s risk because combining assets that don’t move exactly together lowers the portfolio’s variance (the mathematical measure of risk). The more assets you add and the lower their correlations, the more the individual “ups and downs” cancel each other out — but you cannot eliminate market-wide (systematic) risk.

Why, in plain math
- Risk for one asset is measured by its variance (or standard deviation).  
- For a two‑asset portfolio with weights w1 and w2, variances σ1^2 and σ2^2, and correlation ρ between returns, the portfolio variance is

  Var(portfolio) = w1^2 σ1^2 + w2^2 σ2^2 + 2 w1 w2 ρ σ1 σ2.

  The last term (with ρ) is the covariance part that reflects how similarly the two assets move.

What this shows:
- If ρ < 1 (the assets are not perfectly positively correlated), the cross term reduces portfolio variance compared with simply averaging the two individual variances.  
- If ρ = 0 (uncorrelated), the 

> **When to use which?** Deterministic compliance text → low temperature. Brainstorming marketing → high temperature. Hard multi-step analysis → a reasoning model at higher effort (slower, pricier, but deeper). Picking wrong wastes money *and* quality.

## 8. Structured outputs with Pydantic — the headline skill ⭐

So far every answer was **free text**. Production systems need **structured, machine-readable** data they can route, store, and act on. Asking a model to "return JSON" is fragile (it adds prose, breaks quotes, drifts schema). Instead we define a **Pydantic schema** and let the provider **guarantee** the output matches it.

**Business scenario — cross-industry support triage.** An inbound message could be from banking, healthcare, e-commerce, insurance... The intake system must classify each into a fixed schema so it can auto-route to the right queue and dashboard.

In [20]:
from pydantic import BaseModel, Field
from typing import Literal, List

class SupportTicket(BaseModel):
    """Structured triage for any inbound customer message."""

    category: Literal["billing", "technical", "complaint", "inquiry", "fraud_or_safety", "cancellation"] = Field(
        description="Primary intent of the message")
    
    urgency: Literal["low", "medium", "high", "critical"] = Field(description="How fast a human must act")

    sentiment: Literal["positive", "neutral", "negative", "angry"]

    industry: str = Field(description="Best-guess industry, e.g. 'banking', 'healthcare', 'e-commerce'")

    entities: List[str] = Field(description="Key entities: order ids, amounts, names, dates")

    summary: str = Field(description="One-sentence summary")
    
    recommended_action: str = Field(description="Suggested next step for the support agent")

### 8a · OpenAI `responses.parse()` — guaranteed schema → a Pydantic object

In [21]:
msg = ("My credit card was charged ₹18,250 TWICE for the same order #A4821 this morning. "
       "I need one charge refunded today — this is unacceptable.")

resp = openai_client.responses.parse(
    model=M["openai_reason"],
    input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
           {"role": "user", "content": msg}],
    text_format=SupportTicket,
)
ticket = resp.output_parsed          # <-- a real, validated SupportTicket instance
print(type(ticket).__name__, "->")
print(ticket.model_dump_json(indent=2))

SupportTicket ->
{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "Order #A4821",
    "₹18,250",
    "charged twice",
    "this morning",
    "credit card"
  ],
  "summary": "Customer reports their credit card was charged ₹18,250 twice for order #A4821 and demands one charge refunded today.",
  "recommended_action": "Urgently verify payment records for order #A4821, confirm the duplicate charge, and initiate an immediate refund for one of the charges. Inform the customer of the refund transaction ID and expected posting time, apologize, and escalate to billing/payments team if refund cannot be completed immediately. If additional verification is needed, request last 4 digits of the card and a screenshot of the bank statement."
}


### 8b · Classify a batch of diverse, multi-industry messages

In [24]:
messages = [
    "The MRI report upload on your patient portal keeps failing with error 500. I need it before my appointment tomorrow.",   # healthcare
    "I got an OTP I never requested and someone tried to log into my account. Please block it NOW.",                          # fraud / banking
    "Just wanted to say your delivery partner was super polite and my package arrived a day early. Thank you!",               # e-commerce / positive
    "I want to cancel my annual insurance policy renewal scheduled next week and get a pro-rata refund.",                     # insurance
    "What are your customer-care hours on public holidays?",                                                                  # generic inquiry
]

rows = []
for m in messages:
    r = openai_client.responses.parse(
        model=M["openai_reason"],
        input=[{"role": "system", "content": "Classify the customer message into the SupportTicket schema."},
               {"role": "user", "content": m}],
        text_format=SupportTicket,
    )
    t = r.output_parsed
    rows.append({"message": m, "industry": t.industry, "category": t.category, "urgency": t.urgency,
                 "sentiment": t.sentiment, "action": t.recommended_action[:45]})

pd.DataFrame(rows)

,message,industry,category,urgency,sentiment,action
0,The MRI report upload on your patient portal k...,healthcare,technical,high,negative,Escalate to engineering/IT to investigate the
1,I got an OTP I never requested and someone tri...,online services,fraud_or_safety,critical,angry,Immediately suspend/block the account and inv
2,Just wanted to say your delivery partner was s...,e-commerce,inquiry,low,positive,Send a thank-you reply and forward the positi
3,I want to cancel my annual insurance policy re...,insurance,cancellation,high,neutral,"Verify customer identity and policy details,"
4,What are your customer-care hours on public ho...,customer service,inquiry,low,neutral,Provide the official customer‑care hours for


> Every row is a **validated object** — `ticket.urgency` is always one of the allowed values. You can now route on `urgency == "critical"` with zero string-parsing. *This* is why structured output is the backbone of agents (Module 2 onwards).

## 9. Structured output across providers — who supports what

Native structured-output support varies. Know the fallbacks:

| Provider | Native schema enforcement | Method |
|---|:--:|---|
| OpenAI | ✅ | `responses.parse(text_format=Model)` |
| Google Gemini | ✅ | `GenerateContentConfig(response_schema=Model)` → `resp.parsed` |
| Anthropic Claude | ⚠️ (via tool/prompt) | prompt for JSON, then **validate with Pydantic** |
| Groq (Qwen) | ⚠️ | `response_format={"type": "json_object"}` + Pydantic validate |

The universal safety net is the same everywhere: **parse the text, then `Model.model_validate_json(...)`** so bad output fails loudly instead of silently corrupting downstream systems.

In [25]:
# Gemini — native structured output
g = gemini_client.models.generate_content(
    model=M["gemini"], contents=msg,
    config=types.GenerateContentConfig(
        system_instruction="Classify the customer message into the schema.",
        response_mime_type="application/json", response_schema=SupportTicket),
)
print("Gemini  ->", SupportTicket.model_validate_json(g.text).category)


Gemini  -> billing


In [28]:
g.text

'{"category": "billing", "urgency": "high", "sentiment": "angry", "industry": "e-commerce", "entities": ["₹18,250", "order #A4821"], "summary": "Customer was double-charged for the same order and demands an immediate refund for one of the charges.", "recommended_action": "Investigate the duplicate charge for order #A4821 and process a refund for one of the ₹18,250 transactions immediately."}'

In [29]:

# Claude — no native schema; prompt for JSON then validate with Pydantic
schema_hint = SupportTicket.model_json_schema()
c = anthropic_client.messages.create(
    model=M["anthropic"], max_tokens=500,
    system=f"Return ONLY valid JSON matching this JSON Schema (no prose):\n{schema_hint}",
    messages=[{"role": "user", "content": msg}],
)
import json, re
raw = c.content[0].text
raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()  # strip code fences if present
print("Claude  ->", SupportTicket.model_validate(json.loads(raw)).category)

Claude  -> billing


In [31]:
print(raw)

{
  "category": "billing",
  "urgency": "high",
  "sentiment": "angry",
  "industry": "e-commerce",
  "entities": [
    "₹18,250",
    "Order #A4821",
    "duplicate charge",
    "this morning"
  ],
  "summary": "Customer was double-charged ₹18,250 for order #A4821 and demands immediate refund.",
  "recommended_action": "Immediately verify duplicate charge on order #A4821, process refund for one charge within 2 hours, and confirm refund status to customer via email/phone."
}


## 10. Cost & latency at scale

The data behind a vendor decision. Plug in current per-million-token prices and estimate the **monthly** bill for a realistic workload.

In [32]:
# Approx USD per 1M tokens — REPLACE with current pricing before quoting a customer.
pricing = {
    "gpt-4.1-mini":           {"in": 0.40, "out": 1.60},
    "gpt-5-mini":             {"in": 1.10, "out": 4.40},
    "claude-haiku-4-5":       {"in": 0.80, "out": 4.00},
    "gemini-2.5-flash":       {"in": 0.15, "out": 0.60},
    "qwen/qwen3.8-27b":       {"in": 0.80, "out": 4.00},
}

# Workload: 50,000 support tickets/day, ~250 input + ~150 output tokens each.
TICKETS_PER_DAY, IN_TOK, OUT_TOK = 50_000, 250, 150
days = 30
rows = []
for model, p in pricing.items():
    daily_in  = TICKETS_PER_DAY * IN_TOK  / 1_000_000
    daily_out = TICKETS_PER_DAY * OUT_TOK / 1_000_000
    monthly = (daily_in * p["in"] + daily_out * p["out"]) * days
    rows.append({"Model": model, "$/1M in": p["in"], "$/1M out": p["out"], "Est. $/month": round(monthly, 2)})

print(f"Workload: {TICKETS_PER_DAY:,} tickets/day × {days} days  ({IN_TOK} in + {OUT_TOK} out tokens each)\n")
print(pd.DataFrame(rows).sort_values("Est. $/month").to_string(index=False))

Workload: 50,000 tickets/day × 30 days  (250 in + 150 out tokens each)

           Model  $/1M in  $/1M out  Est. $/month
gemini-2.5-flash     0.15       0.6        191.25
    gpt-4.1-mini     0.40       1.6        510.00
claude-haiku-4-5     0.80       4.0       1200.00
qwen/qwen3.8-27b     0.80       4.0       1200.00
      gpt-5-mini     1.10       4.4       1402.50


> The cheapest model per token isn't automatically the right choice — weigh it against the **quality** you measured in §6 and each model's **caution** behaviour. Cost is one axis of three.

## 11. Production pattern: multi-provider fallback 🛡️ *(Black-Friday reliability)*

**Business scenario.** It's the peak-sale spike. Your primary provider starts returning rate-limit / 5xx errors. You **cannot** drop customer messages on the floor. A fallback chain tries providers in order and returns the first success — so a single provider's bad day doesn't take you down.

In [ ]:
def classify_with_fallback(message, order=("OpenAI", "Groq", "Anthropic")):
    """Try providers in order; return the first successful response (+ which provider served it)."""
    last_error = None
    for name in order:
        try:
            out = PROVIDERS[name](
                "Reply in ONE sentence: state the category and urgency of this support message.", message)
            return {"served_by": name, "model": out["model"], "latency": out["latency"], "text": out["text"]}
        except Exception as e:
            last_error = f"{name} failed: {e}"
            print(f"  ⚠️ {last_error} → falling back...")
    raise RuntimeError(f"All providers failed. Last error: {last_error}")

result = classify_with_fallback("URGENT: my account shows a ₹0 balance but I deposited ₹50,000 this morning!")
print(f"\n✅ Served by {result['served_by']} ({result['model']}, {result['latency']}s):\n{result['text']}")

## 12. Conclusion & what's next

| You built | Why it matters |
|---|---|
| 4 native calls + unified wrappers | One interface over four very different SDKs |
| Temperature vs. reasoning effort | The two control knobs, finally untangled |
| **Pydantic structured outputs** | Validated objects, not fragile JSON-in-text — the backbone of agents |
| Cross-provider structured output | Know each provider's native support and the universal validate-it fallback |
| Cost/latency model | Turn "which provider?" into a number |
| Fallback chain | One provider's outage ≠ your outage |

**Next — Lab 1.2: Prompting Techniques & Structured Outputs.** From raw SDK calls to LangChain's `ChatPromptTemplate` — zero- and few-shot prompting, Chain-of-Thought, Tree-of-Thought, and structured outputs inside a chain.